# AGORA z0 Pipeline Runner Only

This notebook is intentionally lightweight: configure one dataset or all datasets, then run `AGORA_parallel_DM_EM_projection_pipeline.py`. It does not contain result plotting cells.

In [ ]:
from pathlib import Path
import sys
import importlib

ROOT = Path('/home/zhaozhang/local/AGORA_work/AGORA_Data/z0')
PIPELINE = ROOT / 'AGORA_parallel_DM_EM_projection_pipeline.py'

if not PIPELINE.exists():
    raise FileNotFoundError(f'Missing pipeline script: {PIPELINE}')

sys.path.insert(0, str(ROOT))
import AGORA_parallel_DM_EM_projection_pipeline as agora
agora = importlib.reload(agora)

print('ROOT    =', ROOT)
print('PIPELINE=', PIPELINE)

In [ ]:
CODE_ALIASES = {
    'ARTI': 'ART-I', 'ART-I': 'ART-I', 'ART': 'ART-I',
    'Enzo': 'ENZO', 'ENZO': 'ENZO',
    'AREPO': 'AREPO',
    'GADGET3': 'GADGET-3', 'GADGET-3': 'GADGET-3',
    'GEAR': 'GEAR',
    'CHANGA': 'CHANGA',
    'G4Cal_Pablo': 'GADGET-4', 'G4CAL_PABLO': 'GADGET-4', 'G4Cal': 'GADGET-4',
}
FOLDER_ALIASES = {
    'ART-I': 'ARTI',
    'ENZO': 'Enzo',
    'AREPO': 'AREPO',
    'GADGET-3': 'GADGET3',
    'GADGET-4': 'G4Cal_Pablo',
    'GEAR': 'GEAR',
    'CHANGA': 'CHANGA',
}

def normalize_user_code(code):
    if code not in CODE_ALIASES:
        raise ValueError(f'Unknown DATASET_CODE={code!r}; choose one of {sorted(CODE_ALIASES)}')
    return CODE_ALIASES[code]

def folder_name_for_code(input_code, normalized):
    if str(input_code).upper() in {'G4CAL_PABLO', 'G4CAL', 'G4CAL-PABLO'}:
        return 'G4Cal_Pablo'
    return FOLDER_ALIASES[normalized]

def is_changa_main_file(path):
    if not path.is_file() or not path.name.startswith('ncal-'):
        return False
    parts = path.name.split('.')
    return len(parts) == 2 and parts[-1].isdigit()

def candidate_snapshots_for_code(code, root=ROOT):
    normalized = normalize_user_code(code)
    folder = root / folder_name_for_code(code, normalized)
    if not folder.exists():
        raise FileNotFoundError(f'Missing data folder for {normalized}: {folder}')
    if normalized == 'ART-I':
        candidates = sorted(folder.glob('*.d'))
    elif normalized == 'ENZO':
        candidates = sorted(p for p in folder.glob('RD*/RD*') if p.is_file() and p.name == p.parent.name)
    elif normalized == 'AREPO':
        candidates = sorted(p for p in folder.glob('snap_*.hdf5') if '.hsml.' not in p.name and '.kdtree' not in p.name)
    elif normalized in {'GADGET-3', 'GADGET-4'}:
        candidates = sorted(folder.glob('snapshot_*/*.0.hdf5')) + sorted(folder.glob('snapshot_*.hdf5'))
    elif normalized == 'GEAR':
        candidates = sorted(p for p in list(folder.glob('snapshot_*.hdf5')) + list(folder.glob('*.hdf5')) if '.hsml.' not in p.name)
    elif normalized == 'CHANGA':
        preferred = sorted(p for p in folder.iterdir() if is_changa_main_file(p))
        fallback = sorted(p for p in folder.iterdir() if p.is_file() and not p.name.startswith('.') and p.name not in {'wget-log', 'robots.txt.tmp'} and not any(p.name.endswith(s) for s in ['.HII', '.massform', '.Metalsdot', '.ESNRate', '.kdtree']))
        candidates = preferred or fallback
    else:
        candidates = []
    return normalized, folder, candidates

def discover_dataset(code, root=ROOT, index=0):
    normalized, folder, candidates = candidate_snapshots_for_code(code, root)
    if not candidates:
        raise FileNotFoundError(f'No snapshot candidate found for {normalized} in {folder}')
    return {'code': normalized, 'folder': folder, 'snapshot': candidates[index], 'candidate_count': len(candidates), 'all_candidates': candidates}

for code in ['ARTI', 'Enzo', 'AREPO', 'GADGET3', 'GEAR', 'CHANGA', 'G4Cal_Pablo']:
    try:
        selected = discover_dataset(code)
        print(f'{code:12s} -> {selected["snapshot"]}')
    except Exception as exc:
        print(f'{code:12s} -> MISSING: {exc}')

## Run Configuration

In [ ]:
# Choose one mode.
RUN_ALL_CODES = False
DATASET_CODE = 'G4Cal_Pablo'   # ARTI, Enzo, AREPO, GADGET3, GEAR, CHANGA, G4Cal_Pablo
ALL_CODES = ['ARTI', 'Enzo', 'AREPO', 'GADGET3', 'GEAR', 'CHANGA', 'G4Cal_Pablo']

MANUAL_SNAPSHOT = None         # Path('/custom/snapshot') or None

N_LOS = 20000
N_JOBS = 30
RANDOM_OBSERVERS = 128
S_MAX_KPC = 100
DS_KPC = 0.25
R_SUN_KPC = 8.2

VELOCITY_REFERENCE_SOURCE = 'stars'
VELOCITY_REFERENCE_RADIUS_KPC = 30.0

ISM_R_KPC = 20.0
ISM_ABS_Z_KPC = 5.0
HOT_TMIN_K = 1.0e6

PROJECTION_BOX_KPC = 40
PROJECTION_NPIX = 512
PROJECTION_MAX_ELEMENTS = 100_000_000
PROJECTION_QUIVER_STEP = 12
PROJECTION_QUIVER_SCALE = 2200
PROJECTION_QUIVER_WIDTH = 0.0022
PROJECTION_QUIVER_ALPHA = 0.75

IONIZATION_MODE = 'auto'
OUTPUT_BASE = ROOT / 'parallel_outputs'
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

In [ ]:
def run_one_dataset(dataset_code):
    if MANUAL_SNAPSHOT is None:
        selected = discover_dataset(dataset_code)
        code = selected['code']
        snapshot = selected['snapshot']
    else:
        code = normalize_user_code(dataset_code)
        snapshot = Path(MANUAL_SNAPSHOT)

    outdir = OUTPUT_BASE / folder_name_for_code(dataset_code, code)
    outdir.mkdir(parents=True, exist_ok=True)

    print('\n' + '=' * 90)
    print('DATASET_CODE =', dataset_code)
    print('CODE         =', code)
    print('SNAPSHOT     =', snapshot)
    print('OUTDIR       =', outdir)

    args = [
        '--snapshot', str(snapshot),
        '--code', code,
        '--integration-backend', 'auto',
        '--outdir', str(outdir),
        '--unit-base', 'auto',
        '--n-los', str(N_LOS),
        '--particle-interpolation', 'auto',
        '--ionization-mode', IONIZATION_MODE,
        '--n-jobs', str(N_JOBS),
        '--chunk-los', '64',
        '--s-max-kpc', str(S_MAX_KPC),
        '--ds-kpc', str(DS_KPC),
        '--R-sun-kpc', str(R_SUN_KPC),
        '--center-mode', 'stellar_com',
        '--disk-normal-source', 'stars',
        '--velocity-reference-source', VELOCITY_REFERENCE_SOURCE,
        '--velocity-reference-radius-kpc', str(VELOCITY_REFERENCE_RADIUS_KPC),
        '--ism-R-kpc', str(ISM_R_KPC),
        '--ism-abs-z-kpc', str(ISM_ABS_Z_KPC),
        '--hot-Tmin-K', str(HOT_TMIN_K),
        '--make-mollweide',
        '--make-hot-em-diagnostics',
        '--make-projections',
        '--projection-box-kpc', str(PROJECTION_BOX_KPC),
        '--projection-npix', str(PROJECTION_NPIX),
        '--projection-max-elements', str(PROJECTION_MAX_ELEMENTS),
        '--projection-quiver-step', str(PROJECTION_QUIVER_STEP),
        '--projection-quiver-scale', str(PROJECTION_QUIVER_SCALE),
        '--projection-quiver-width', str(PROJECTION_QUIVER_WIDTH),
        '--projection-quiver-alpha', str(PROJECTION_QUIVER_ALPHA),
        '--random-observers', str(RANDOM_OBSERVERS),
    ]
    agora.main(args)
    return outdir

if RUN_ALL_CODES:
    completed = {code: run_one_dataset(code) for code in ALL_CODES}
else:
    completed = {DATASET_CODE: run_one_dataset(DATASET_CODE)}

completed